GAN

In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import DataLoader, TensorDataset

In [16]:
# Simple Generator
class Generator(nn.Module):
    def __init__(self, latent_dim=32, output_dim=64):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 256),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 512),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, output_dim),
            nn.Tanh()
        )
    
    def forward(self, z):
        return self.model(z)

# Simple Discriminator
class Discriminator(nn.Module):
    def __init__(self, input_dim=64):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.model(x)



In [110]:
# Load data
def load_data():
    #real_vectors = np.load("/Users/angzeng/GitHub/wisepanda/algorithm/dataset/Bamboo500.npy")
    #real_vectors = np.load("/Users/angzeng/GitHub/wisepanda/algorithm/dataset/interference_data_bamboo.npy")
    real_vectors2 = np.load("/Users/angzeng/GitHub/wisepanda/algorithm/dataset/vector_real_118_patch.npy")

    top_vectors = np.squeeze(real_vectors2[:, 2:3, :]) 
    bottom_vectors = np.squeeze(real_vectors2[:, 3:4, :])
    real_vectors3 = np.concatenate([top_vectors, bottom_vectors], axis=0)

    # Normalize data to [-1, 1] range to match generator output
    data = torch.tensor(real_vectors3, dtype=torch.float32)
    data = (data - data.min()) / (data.max() - data.min()) * 2 - 1
    return data

In [111]:
def train_gan():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on device: {device}")
    
    # Load data
    real_data = load_data()
    dataloader = DataLoader(TensorDataset(real_data), batch_size=32, shuffle=True)
    
    # Initialize models
    G = Generator().to(device)
    D = Discriminator().to(device)
    
    # Different learning rates for G and D
    g_optimizer = optim.Adam(G.parameters(), lr=0.0002, betas=(0.5, 0.999))
    d_optimizer = optim.Adam(D.parameters(), lr=0.0001, betas=(0.5, 0.999))  # Slower for D
    
    criterion = nn.BCELoss()
    
    # Training loop with improved balance
    epochs = 100
    d_steps = 1  # Train discriminator every step
    g_steps = 2  # Train generator twice per discriminator update
    
    for epoch in range(epochs):
        epoch_d_loss = 0
        epoch_g_loss = 0
        num_batches = 0
        
        for batch_idx, (real_vectors,) in enumerate(dataloader):
            batch_size = real_vectors.size(0)
            real_vectors = real_vectors.to(device)
            num_batches += 1
            
            # Train Discriminator (less frequently)
            if batch_idx % d_steps == 0:
                d_optimizer.zero_grad()
                
                # Real data
                real_labels = torch.ones(batch_size, 1).to(device) * 0.9  # Label smoothing
                real_output = D(real_vectors)
                d_loss_real = criterion(real_output, real_labels)
                
                # Fake data
                z = torch.randn(batch_size, 32).to(device)
                fake_vectors = G(z).detach()
                fake_labels = torch.zeros(batch_size, 1).to(device) + 0.1  # Label smoothing
                fake_output = D(fake_vectors)
                d_loss_fake = criterion(fake_output, fake_labels)
                
                d_loss = (d_loss_real + d_loss_fake) / 2
                d_loss.backward()
                d_optimizer.step()
                epoch_d_loss += d_loss.item()
            
            # Train Generator (more frequently)
            for _ in range(g_steps):
                g_optimizer.zero_grad()
                
                z = torch.randn(batch_size, 32).to(device)
                fake_vectors = G(z)
                fake_output = D(fake_vectors)
                
                # Generator wants discriminator to output 1 (real)
                real_labels = torch.ones(batch_size, 1).to(device)
                g_loss = criterion(fake_output, real_labels)
                
                g_loss.backward()
                g_optimizer.step()
                epoch_g_loss += g_loss.item()
        
        # Print progress
        if epoch % 10 == 0:
            avg_d_loss = epoch_d_loss / num_batches
            avg_g_loss = epoch_g_loss / (num_batches * g_steps)
            print(f'Epoch [{epoch}/{epochs}] D_loss: {avg_d_loss:.4f} G_loss: {avg_g_loss:.4f}')
            
            # Check if losses are balanced
            if avg_d_loss < 0.3:
                print("Warning: Discriminator might be too strong!")
            if avg_g_loss > 2.0:
                print("Warning: Generator might be struggling!")
    
    return G, D

In [112]:
# Generate new samples
def generate_samples(generator, num_samples=100):
    device = next(generator.parameters()).device
    generator.eval()
    with torch.no_grad():
        z = torch.randn(num_samples, 32).to(device)
        generated = generator(z)
        return generated.cpu().numpy()

In [113]:
generator, discriminator = train_gan()
# Generate samples
samples = generate_samples(generator, 6000)
print(f"Generated {len(samples)} samples of shape {samples.shape}")

# Save generated data
np.save("generated_vectors_GAN(236).npy", samples)
print("Generated samples saved to 'generated_vectors_GAN(236).npy'")

Training on device: cpu
Epoch [0/100] D_loss: 0.6706 G_loss: 0.6479
Epoch [10/100] D_loss: 0.6938 G_loss: 0.6794
Epoch [20/100] D_loss: 0.6958 G_loss: 0.6996
Epoch [30/100] D_loss: 0.6926 G_loss: 0.7010
Epoch [40/100] D_loss: 0.6918 G_loss: 0.6991
Epoch [50/100] D_loss: 0.6937 G_loss: 0.6979
Epoch [60/100] D_loss: 0.6927 G_loss: 0.6748
Epoch [70/100] D_loss: 0.6923 G_loss: 0.7067
Epoch [80/100] D_loss: 0.6943 G_loss: 0.6714
Epoch [90/100] D_loss: 0.6931 G_loss: 0.7005
Generated 6000 samples of shape (6000, 64)
Generated samples saved to 'generated_vectors_GAN(236).npy'


In [108]:
class FractureCurveGenerator:
    def __init__(self, count_fiber, K_Ⅲ, len_x, ce_rate, erosion_epoch, data_file="generated_vectors_GAN.npy", save_path='seriesgan_data_6000.npy'):
        self.count_fiber = count_fiber
        self.K_Ⅲ = K_Ⅲ
        self.len_x = len_x
        self.ce_rate = ce_rate
        self.erosion_epoch = erosion_epoch
        self.save_path = save_path
        self.data_file = data_file
        
        # Load pre-generated data
        self.generated_samples = np.load(data_file)
        self.current_index = 0  # Track current data index being used
        
        print(f"Loaded {len(self.generated_samples)} samples from {data_file}")
        print(f"Sample shape: {self.generated_samples.shape}")

    def create_fiber_line(self): 
        '''Description: This function is used to generate the fracture curve'''
        # Get a sample from pre-loaded data
        if self.current_index >= len(self.generated_samples):
            # If all data is used up, start over with cycling
            self.current_index = 0
            print("Warning: Reusing data samples as all samples have been used.")
        
        # Get current sample
        sample = self.generated_samples[self.current_index]
        self.current_index += 1
        
        # Convert to list format
        if isinstance(sample, np.ndarray):
            y_list = sample.tolist()
        else:
            y_list = [sample] if not isinstance(sample, list) else sample
            
        return y_list

    def reset_data_index(self):
        '''Reset the data index to start from the beginning'''
        self.current_index = 0

    def get_random_fiber_line(self):
        '''Get a random sample from the loaded data'''
        random_index = np.random.randint(0, len(self.generated_samples))
        sample = self.generated_samples[random_index]
        
        if isinstance(sample, np.ndarray):
            y_list = sample.tolist()
        else:
            y_list = [sample] if not isinstance(sample, list) else sample
            
        return y_list

    def relu(self, x):
        '''Calculate the ReLU activation function'''
        return np.maximum(0, x)

    def erosion_new(self, list):
        '''Generate a single step of erosion on the curve'''
        new_list = []
        for i in range(len(list)):
            # Determine left and right neighbor values
            if i == 0:
                left_fiber = list[i]
                if len(list) == 1:
                    right_fiber = list[i]
                else:
                    right_fiber = list[i+1]
            elif i == len(list)-1:
                left_fiber = list[i-1]
                right_fiber = list[i]
            else:
                left_fiber = list[i-1]
                right_fiber = list[i+1]
            
            # Calculate erosion for each fiber element
            structural_area = self.relu(list[i]-left_fiber)+self.relu(list[i]-right_fiber)  # Calculate structural area
            erosion_fiber = list[i]-structural_area*self.ce_rate
            new_list.append(erosion_fiber)

        return new_list

    def erosion_with_epoch(self, list, epoch): 
        '''Apply erosion process for multiple epochs'''
        for i in range(epoch):
            list = self.erosion_new(list)
        return list

    def floor_list(self, list, floor):
        '''Adjust the list values to a specified floor level'''
        min_value = min(list) + floor
        adjusted_list = [x - min_value for x in list]
        return adjusted_list

    def revers_list(self, list, floor): 
        '''Reverse the list values and adjust to floor level'''
        inverted_list = [-x for x in list]
        return self.floor_list(inverted_list, floor)

    def get_top_erosion_fiber(self, list, epoch, floor): 
        '''Generate the top erosion fiber through reverse processing'''
        reversed_list = self.revers_list(list, 0)
        erosioned_list = self.erosion_with_epoch(reversed_list, epoch)
        adjusted_list = self.revers_list(erosioned_list, -floor)
        return adjusted_list

    def fibers_resize(self, list, num):
        '''Resize fibers to specified number of points using interpolation'''
        # Create new indices with specified length
        new_indices = np.linspace(0, len(list) - 1, num=num)
        # Use linear interpolation
        resampled_list = np.interp(new_indices, np.arange(len(list)), list)
        resampled_list = [64*x/self.count_fiber for x in resampled_list]
        resampled_list = [round(x, 4) for x in resampled_list]
        return resampled_list

    def get_pair_fibers(self, use_random=False):
        '''Generate a pair of erosion fibers (top and bottom)'''
        if use_random:
            fracture_list = self.get_random_fiber_line()
        else:
            fracture_list = self.create_fiber_line()
            
        erosion_list = self.erosion_with_epoch(fracture_list, self.erosion_epoch)
        erosion_list = self.floor_list(erosion_list, 0)
        top_erosion_list = self.get_top_erosion_fiber(fracture_list, self.erosion_epoch, 0)
        return self.fibers_resize(erosion_list, 64), self.fibers_resize(top_erosion_list, 64)

    def get_fracture_curves(self, data_amount):
        '''Generate processed fracture curve data for training'''
        data_list = []
        array_zero = np.zeros(64)
        
        # Reset index to start from the beginning
        self.reset_data_index()
        
        # Generate basic fracture curve pairs
        for i in range(data_amount):
            a, b = self.get_pair_fibers()
            vector_edge_top = np.array(a)
            vector_edge_bottom = np.array(b)
            list_top_bottom = [array_zero, array_zero, vector_edge_top, vector_edge_bottom]
            data_list.append(list_top_bottom)

        all_data_list = np.array(data_list)
        np.save(self.save_path, all_data_list)
        print(f"Saved {len(all_data_list)} processed samples to '{self.save_path}'")
        return all_data_list

In [109]:
generator = FractureCurveGenerator(
    count_fiber=64,  
    K_Ⅲ=1.0,
    len_x=3,
    ce_rate=0.02,
    erosion_epoch=500,
    data_file="generated_vectors_GAN(236).npy",
    save_path="generated_vectors_GAN_0.02_6000(236).npy"
)

processed_data = generator.get_fracture_curves(6000)

Loaded 6000 samples from generated_vectors_GAN(236).npy
Sample shape: (6000, 64)


KeyboardInterrupt: 

In [56]:
generator = FractureCurveGenerator(
    count_fiber=64,  
    K_Ⅲ=1.0,
    len_x=3,
    ce_rate=0.02,
    erosion_epoch=500,
    data_file="generated_vectors_GAN(1350).npy",
    save_path="generated_vectors_GAN_0.02_6000(1350).npy"
)

processed_data = generator.get_fracture_curves(6000)

Loaded 6000 samples from generated_vectors_GAN(1350).npy
Sample shape: (6000, 64)
Saved 6000 processed samples to 'generated_vectors_GAN_0.02_6000(1350).npy'


In [47]:
generator = FractureCurveGenerator(
    count_fiber=64,  # 根据generated_vectors_GAN.npy的数据维度设置
    K_Ⅲ=1.0,
    len_x=3,
    ce_rate=0.02,
    erosion_epoch=500,
    data_file="generated_vectors_GAN.npy",
    save_path="generated_vectors_GAN_0.02_6000.npy"
)

processed_data = generator.get_fracture_curves(6000)

Loaded 6000 samples from generated_vectors_GAN.npy
Sample shape: (6000, 64)
Saved 6000 processed samples to 'generated_vectors_GAN_0.02_6000.npy'


In [22]:
generator = FractureCurveGenerator(
    count_fiber=64,  
    K_Ⅲ=1.0,
    len_x=3,
    ce_rate=0.02,
    erosion_epoch=500,
    data_file="generated_vectors_GAN(1114).npy",
    save_path="generated_vectors_GAN_0.02_6000(1114).npy"
)

processed_data = generator.get_fracture_curves(6000)

Loaded 6000 samples from generated_vectors_GAN(1114).npy
Sample shape: (6000, 64)
Saved 6000 processed samples to 'generated_vectors_GAN_0.02_6000(1114).npy'


In [ ]:
generator = FractureCurveGenerator(
    count_fiber=64,  
    K_Ⅲ=1.0,
    len_x=3,
    ce_rate=0.01,
    erosion_epoch=500,
    data_file="generated_vectors_GAN.npy",
    save_path="generated_vectors_GAN_0.01_6000.npy"
)

processed_data = generator.get_fracture_curves(6000)

Loaded 6000 samples from generated_vectors_GAN.npy
Sample shape: (6000, 64)
Saved 6000 processed samples to 'generated_vectors_GAN_0.01_6000.npy'


In [49]:
def perturb_curve(y, noise_level=0.02):
    """
    Simplified curve perturbation function to add realistic roughness
    
    Args:
        y: Original curve data (1D numpy array, float32 format)
        noise_level: Overall noise strength (0.01-0.1 recommended)
    
    Returns:
        Perturbed curve with added roughness (same format as input)
    """
    # Ensure input is numpy array and preserve original dtype
    y = np.asarray(y)
    original_dtype = y.dtype
    
    x = np.linspace(0, 1, len(y))
    
    # 1. Basic Gaussian noise
    basic_noise = np.random.normal(0, noise_level, len(y))
    
    # 2. High-frequency noise for fine-scale variations
    high_freq = np.random.normal(0, noise_level * 0.5, len(y))
    
    # 3. Mid-frequency periodic perturbation
    mid_freq = np.sin(2*np.pi*5*x) * noise_level * 2
    
    # 4. Low-frequency drift for long-term variations
    low_freq = np.cumsum(np.random.normal(0, noise_level/10, len(y)))
    low_freq = low_freq - np.linspace(low_freq[0], low_freq[-1], len(low_freq))
    
    # 5. Nonlinear perturbation: y_new = y + β * sin(γ * y) * noise
    nonlinear_noise = np.random.normal(0, noise_level * 0.8, len(y))
    nonlinear = 0.03 * np.sin(2 * y) * nonlinear_noise
    
    # Combine all perturbations
    y_perturbed = y + basic_noise + high_freq + mid_freq + low_freq + nonlinear
    
    # Preserve original dtype (float32)
    return y_perturbed.astype(original_dtype)

In [50]:
generated_samples = np.load("generated_vectors_GAN.npy")
perturbed_samples = [perturb_curve(sample, noise_level=0.12) for sample in generated_samples]
np.save("generated_vectors_GAN_perturbed.npy", perturbed_samples)

In [51]:
generator = FractureCurveGenerator(
    count_fiber=64,  
    K_Ⅲ=1.0,
    len_x=3,
    ce_rate=0.02,
    erosion_epoch=500,
    data_file="generated_vectors_GAN_perturbed.npy",
    save_path="generated_vectors_GAN_perturbed_0.02_6000.npy"
)

processed_data = generator.get_fracture_curves(6000)

Loaded 6000 samples from generated_vectors_GAN_perturbed.npy
Sample shape: (6000, 64)
Saved 6000 processed samples to 'generated_vectors_GAN_perturbed_0.02_6000.npy'
